# Conditional Transformer for SMILES Generation

本 Notebook 演示如何从 `lam_abs`, `lam_em`, `eps` 三个属性生成 SMILES 序列。
包含：
1. 属性标准化预处理
2. 数据加载和 Dataset 定义
3. Prop2SmilesModel 模型定义（属性自编码 + Transformer Decoder）
4. 模型训练
5. 推理生成 SMILES

In [2]:
# 1. 安装与导入
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from utils import split, validity, sample   
from pretrain_trfm import PositionalEncoding


ImportError: attempted relative import with no known parent package

In [145]:
import pickle
with open(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

print('%10' in vocab.stoi)   
print('0' in vocab.stoi)   
print('@' in vocab.stoi)   

True
False
True


In [102]:
# 2. 属性标准化预处理
# df = pd.read_csv(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\data_train.csv')  
# scaler = StandardScaler()
# props = df[['lam_abs','lam_em','eps']].values
# props_scaled = scaler.fit_transform(props)
# df_scaled = df.copy()
# df_scaled[['lam_abs','lam_em','eps']] = props_scaled
# df_scaled.to_csv(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\data_scaled.csv', index=False)
# joblib.dump(scaler, r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\data_scaled.csv')
# print('Saved data_scaled.csv and prop_scaler.pkl')
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

# 1. 读取数据
df = pd.read_csv(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\data_train.csv')  

# 2. 标准化处理
scaler = StandardScaler()
props = df[['lam_abs','lam_em','eps']].values
props_scaled = scaler.fit_transform(props)

# 3. 保存标准化后的数据（CSV格式）
df_scaled = df.copy()
df_scaled[['lam_abs','lam_em','eps']] = props_scaled
df_scaled.to_csv(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\data_scaled.csv', index=False)

# 4. 单独保存scaler对象（使用.pkl扩展名）
joblib.dump(scaler, r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\prop_scaler.pkl')

print('标准化数据已保存为 data_scaled.csv，标准化器已保存为 prop_scaler.pkl')



标准化数据已保存为 data_scaled.csv，标准化器已保存为 prop_scaler.pkl


In [146]:
# 3. Dataset 定义
import pickle
from build_vocab import Vocab
from build_vocab import WordVocab
class Prop2SmilesDataset(Dataset):
    def __init__(self, csv_path, smiles_col, prop_cols, vocab_path, seq_len=220):
        df = pd.read_csv(csv_path)
        self.props = torch.tensor(df[prop_cols].values.astype(float), dtype=torch.float)
        self.smiles = df[smiles_col].astype(str).tolist()
        self.seq_len = seq_len
        with open(vocab_path, 'rb') as f:
            self.vocab = pickle.load(f)
    def __len__(self):
        return len(self.props)
    def __getitem__(self, idx):
        prop = self.props[idx]
        sm = self.smiles[idx]
        toks = split(sm).split()
        ids = [self.vocab.stoi.get(t, self.vocab.unk_index) for t in toks]
        seq = [self.vocab.sos_index] + ids + [self.vocab.eos_index]
        if len(seq) < self.seq_len:
            seq += [self.vocab.pad_index] * (self.seq_len - len(seq))
        else:
            seq = seq[:self.seq_len]
        return prop, torch.tensor(seq, dtype=torch.long)

# 实例化测试
ds = Prop2SmilesDataset(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\data_scaled.csv', 'smiles', ['lam_abs','lam_em','eps'], r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\vocab.pkl')
print('Dataset size:', len(ds))

Dataset size: 111


In [122]:
# 4. 模型定义
class Prop2SmilesModel(nn.Module):
    def __init__(self, prop_dim=3, latent_dim=128, vocab_size=5000,
                 hidden_size=256, n_head=4, n_dec_layers=4,
                 seq_len=220, prop_recon_weight=0.1, dropout=0.1):
        super().__init__()
        self.prop_encoder = nn.Sequential(
            nn.Linear(prop_dim, latent_dim), nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )
        self.prop_decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim), nn.ReLU(),
            nn.Linear(latent_dim, prop_dim)
        )
        self.memory_len = seq_len // 10
        self.mem_proj = nn.Linear(latent_dim, self.memory_len * hidden_size)
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.pos_enc = PositionalEncoding(hidden_size, dropout, max_len=seq_len)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size, nhead=n_head,
            dim_feedforward=hidden_size*4, dropout=dropout)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=n_dec_layers,
                                             norm=nn.LayerNorm(hidden_size))
        self.out_proj = nn.Linear(hidden_size, vocab_size)
        self.ce = nn.CrossEntropyLoss(ignore_index=0)
        self.mse= nn.MSELoss()
        self.w  = prop_recon_weight
    def forward(self, props, seq):
        B, L = seq.shape
        z = self.prop_encoder(props)
        rp = self.prop_decoder(z)
        Lp = self.mse(rp, props)
        m = self.mem_proj(z).view(B, self.memory_len, -1)
        mem = m.permute(1,0,2)
        emb = self.embedding(seq).permute(1,0,2)
        emb = self.pos_enc(emb)
        mask = nn.Transformer.generate_square_subsequent_mask(L).to(seq.device)
        out = self.decoder(tgt=emb, memory=mem, tgt_mask=mask)
        logits = self.out_proj(out).permute(1,2,0)
        tgt = seq[:,1:].contiguous()
        logits = logits[:,:,:L-1]
        Ls = self.ce(logits, tgt)
        return Ls + self.w*Lp, Ls, Lp

# 实例化测试
model = Prop2SmilesModel(
    prop_dim=3, latent_dim=128, vocab_size=len(ds.vocab),
    hidden_size=256, n_head=4, n_dec_layers=4,
    seq_len=ds.seq_len, prop_recon_weight=0.1
).to('cuda' if torch.cuda.is_available() else 'cpu')
print(model)

Prop2SmilesModel(
  (prop_encoder): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (prop_decoder): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=3, bias=True)
  )
  (mem_proj): Linear(in_features=128, out_features=5632, bias=True)
  (embedding): Embedding(71, 256, padding_idx=0)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_fe

In [147]:
# 5. 训练
device = 'cuda' if torch.cuda.is_available() else 'cpu'
loader = DataLoader(ds, batch_size=32, shuffle=True)
opt = optim.Adam(model.parameters(), lr=1e-4)
for epoch in range(1, 2000):
    model.train()
    total_loss = 0
    for props, seq in loader:
        props, seq = props.to(device), seq.to(device)
        opt.zero_grad()
        loss, Ls, Lp = model(props, seq)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch:02d} | Total Loss: {total_loss/len(loader):.4f}")
torch.save(model.state_dict(), r'D:\workspace\pycharm\smiles-transformer\output\proc2smiles\proc2smiles_data_train_epoch2000.pt')

Epoch 01 | Total Loss: 0.1294
Epoch 02 | Total Loss: 0.1197
Epoch 03 | Total Loss: 0.1108
Epoch 04 | Total Loss: 0.1103
Epoch 05 | Total Loss: 0.1066
Epoch 06 | Total Loss: 0.1031
Epoch 07 | Total Loss: 0.1063
Epoch 08 | Total Loss: 0.0952
Epoch 09 | Total Loss: 0.0979
Epoch 10 | Total Loss: 0.0972
Epoch 11 | Total Loss: 0.0958
Epoch 12 | Total Loss: 0.0973
Epoch 13 | Total Loss: 0.0902
Epoch 14 | Total Loss: 0.0936
Epoch 15 | Total Loss: 0.0878
Epoch 16 | Total Loss: 0.0992
Epoch 17 | Total Loss: 0.0944
Epoch 18 | Total Loss: 0.0959
Epoch 19 | Total Loss: 0.0946
Epoch 20 | Total Loss: 0.0945
Epoch 21 | Total Loss: 0.0963
Epoch 22 | Total Loss: 0.0972
Epoch 23 | Total Loss: 0.0920
Epoch 24 | Total Loss: 0.1026
Epoch 25 | Total Loss: 0.1010
Epoch 26 | Total Loss: 0.0957
Epoch 27 | Total Loss: 0.0962
Epoch 28 | Total Loss: 0.0900
Epoch 29 | Total Loss: 0.0950
Epoch 30 | Total Loss: 0.0986
Epoch 31 | Total Loss: 0.0942
Epoch 32 | Total Loss: 0.0898
Epoch 33 | Total Loss: 0.0934
Epoch 34 |

In [149]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
scaler = joblib.load(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\train\prop_scaler.pkl')
with open(r'D:\workspace\pycharm\smiles-transformer\data\WPT,AJ,ZYM\datasets\vocab.pkl','rb') as f:
    vocab = pickle.load(f)
    vovab_size = len(vocab)
print("词表大小:", len(vocab))
print("示例 token:", list(vocab.stoi.keys())[:10])
print("特殊标记:", vocab.pad_index, vocab.sos_index, vocab.eos_index, vocab.unk_index)

model.load_state_dict(torch.load(r'D:\workspace\pycharm\smiles-transformer\output\proc2smiles\proc2smiles_data_train.pt'))
model.eval()

def generate_smiles(raw_prop, max_len=300):
    prop_scaled = scaler.transform([raw_prop])[0]
    z = model.prop_encoder(torch.tensor(prop_scaled, dtype=torch.float).to(device)[None])
    m = model.mem_proj(z).view(1, model.memory_len, -1).permute(1,0,2)
    seq = [vocab.sos_index]
    for _ in range(max_len):
        inp = torch.tensor([seq], device=device)
        emb = model.embedding(inp).permute(1,0,2)
        emb = model.pos_enc(emb)
        mask = nn.Transformer.generate_square_subsequent_mask(len(seq)).to(device)
        out = model.decoder(tgt=emb, memory=m, tgt_mask=mask)
        logits = model.out_proj(out)[-1,0]
        idx = logits.argmax().item()
        if idx == vocab.eos_index:
            break
        seq.append(idx)
    print("Token indices:", seq)
    print("Tokens:", [vocab.itos[i] if i < len(vocab.itos) else "<OOV>" for i in seq])

    return ''.join(vocab.itos[i] for i in seq[1:])

# 测试
test_raw = [675, 711, 86000]
print('Generated SMILES:', generate_smiles(test_raw))
print("Vocab size:", len(vocab.itos))
print("model.out_proj.weight:", model.out_proj.weight.shape)


词表大小: 45
示例 token: ['<pad>', '<unk>', '<eos>', '<sos>', '<mask>', 'C', '=', '(', ')', 'N']
特殊标记: 0 3 2 1


C:\Users\Admin\AppData\Local\Temp\ipykernel_30780\2880798170.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r'D:\workspace\pycharm\smi

Token indices: [3, 5, 10, 5, 14, 6, 5, 5, 6, 5, 7, 5, 6, 5, 14, 8, 5, 15, 6, 5, 5, 7, 6, 5, 16, 9, 6, 5, 17, 5, 7, 6, 5, 5, 7, 6, 12, 9, 24, 13, 17, 12, 23, 21, 13, 7, 11, 8, 7, 11, 8, 9, 15, 16, 8, 5, 18, 6, 5, 5, 6, 5, 7, 5, 6, 5, 18, 8, 10, 5, 8, 5, 19, 6, 5, 5, 6, 5, 5, 6, 5, 19, 8, 5, 20, 6, 5, 5, 6, 5, 7, 5, 6, 5, 20, 8, 10, 5, 5, 9, 5, 7, 6, 10, 8, 5, 5, 5, 7, 6, 10, 8, 10]
Tokens: ['<sos>', 'C', 'O', 'C', '1', '=', 'C', 'C', '=', 'C', '(', 'C', '=', 'C', '1', ')', 'C', '2', '=', 'C', 'C', '(', '=', 'C', '3', 'N', '=', 'C', '4', 'C', '(', '=', 'C', 'C', '(', '=', '[', 'N', '+', ']', '4', '[', 'B', '-', ']', '(', 'F', ')', '(', 'F', ')', 'N', '2', '3', ')', 'C', '5', '=', 'C', 'C', '=', 'C', '(', 'C', '=', 'C', '5', ')', 'O', 'C', ')', 'C', '6', '=', 'C', 'C', '=', 'C', 'C', '=', 'C', '6', ')', 'C', '7', '=', 'C', 'C', '=', 'C', '(', 'C', '=', 'C', '7', ')', 'O', 'C', 'C', 'N', 'C', '(', '=', 'O', ')', 'C', 'C', 'C', '(', '=', 'O', ')', 'O']
Generated SMILES: COC1=CC=C(C=C1)C2=CC